<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/split_wise2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import heapq

def heap_settler(balances):
    owe, get, txn = [], [], []
    for i,b in enumerate(balances):
        if isinstance(b, tuple):
            b, i = b
        if b > 0:
            heapq.heappush(get, (-b,i))
        elif b < 0:
            heapq.heappush(owe, (b,i))
    while get and owe:
        gb, gi = heapq.heappop(get)
        ob, oi = heapq.heappop(owe)
        if gb < ob:
            txn.append((oi, gi, -ob))
            heapq.heappush(get, (gb-ob,gi))
        elif gb > ob:
            txn.append((oi,gi,-gb))
            heapq.heappush(owe,(ob-gb,oi))
        else:
            txn.append((oi,gi,-ob))
    return txn

In [34]:
txn = heap_settler([-8,5,3,-7,7])
print(txn)
txn = heap_settler([8,-8,7,3,-10])
print(txn)

[(0, 4, 7), (3, 1, 5), (3, 2, 2), (0, 2, 1)]
[(4, 0, 8), (1, 2, 7), (4, 3, 2), (1, 3, 1)]


In [35]:
from itertools import chain, combinations, product
from functools import cache

def min_settler(balances):
    owe, get = [], []
    for i,b in enumerate(balances):
        if b > 0:
            get.append((b,i))
        elif b < 0:
            owe.append((b,i))

    subgroups = max_0sum(tuple(get), tuple(owe))
    txn = []
    for g in subgroups:
        txn += heap_settler(g)
    return txn

powerset = lambda s: chain.from_iterable(combinations(s,r) for r in range(1, len(s)+1))

@cache
def max_0sum(get, owe):
    max_parts = []
    for cre, deb in product(powerset(get), powerset(owe)):
        s = sum(x[0] for x in cre) + sum(x[0] for x in deb)
        if s == 0:
            g_rem = tuple(x for x in get if x not in cre)
            o_rem = tuple(x for x in owe if x not in deb)
            parts = [[*cre,*deb]]
            if g_rem and o_rem:
                if len(g_rem) == 1 and len(o_rem) == 1:
                    parts += [[*g_rem, *o_rem]]
                else:
                    parts += max_0sum(g_rem, o_rem)
            if len(parts) > len(max_parts):
                max_parts = parts
    # print(max_parts)
    return max_parts

In [36]:
txn = min_settler([8,-8,7,3,-10])
print(txn)
txn = heap_settler([+12, +11, +10, +9, -15, -14, -8, -5])
print(txn)
txn = min_settler([+12, +11, +10, +9, -15, -14, -8, -5])
print(txn)

[[(8, 0), (-8, 1)], [(7, 2), (3, 3), (-10, 4)]]
[(1, 0, 8), (4, 2, 7), (4, 3, 3)]
[(4, 0, 12), (5, 1, 11), (6, 2, 8), (7, 3, 5), (4, 3, 3), (5, 2, 2), (5, 3, 1)]
[[(12, 0), (11, 1), (-15, 4), (-8, 6)], [(10, 2), (9, 3), (-14, 5), (-5, 7)]]
[(4, 0, 12), (6, 1, 8), (4, 1, 3), (5, 2, 10), (7, 3, 5), (5, 3, 4)]


In [37]:
balances = [10,-5,-5,-12,7,8,-3,4,-4,-1,-1,-1,3]
txn = heap_settler(balances=balances)
print(txn)
txn = min_settler(balances=balances)
print(txn)

[(3, 0, 10), (1, 5, 5), (2, 4, 5), (8, 7, 4), (6, 5, 3), (3, 12, 2), (9, 4, 1), (10, 4, 1), (11, 12, 1)]
[[(10, 0), (-5, 1), (-5, 2)], [(7, 4), (-3, 6), (-4, 8)], [(3, 12), (-1, 9), (-1, 10), (-1, 11)], [(8, 5), (4, 7), (-12, 3)]]
[(1, 0, 5), (2, 0, 5), (8, 4, 4), (6, 4, 3), (9, 12, 1), (10, 12, 1), (11, 12, 1), (3, 5, 8), (3, 7, 4)]
